# Notebook 33 -- Luyben SBI Training + SBC

Train SNPE_C on the Luyben plant and validate with simulation-based calibration:
1. Train posterior (n_simulations=10k, NSF 192 hidden, 7 transforms).
2. SBC on 500 test cases.
3. KS p-values for all 8 parameters.
4. Save trained posterior to `results/luyben_posterior.pkl`.


In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from cstr_sbi.luyben.priors import box_uniform_8d, PARAM_NAMES
from cstr_sbi.luyben.inference import train_sbi_posterior, sample_posterior
from cstr_sbi.luyben.summaries import compute_summary_statistics
from cstr_sbi.luyben.physics import NOMINAL_INLET, NOMINAL_CTRL_ALL, NOMINAL_Y0

SEED = 42
N_SIM = 10_000  # increase to 20k if SBC fails for >2 params

prior = box_uniform_8d()
print(f'Training SBI with {N_SIM} simulations ...')


In [ ]:
posterior, meta = train_sbi_posterior(
    prior,
    n_simulations=N_SIM,
    seed=SEED,
)
print('Metadata:', meta)

out_path = Path('../results/luyben_posterior.pkl')
out_path.parent.mkdir(exist_ok=True)
import pickle
with open(out_path, 'wb') as f:
    pickle.dump({'posterior': posterior, 'metadata': meta}, f)
print(f'Saved to {out_path}')


## SBC validation

In [ ]:
from scipy.stats import ks_1samp, uniform
import jax
import jax.numpy as jnp
from cstr_sbi.luyben.simulator import simulate_em_window, warm_start_ic, apply_sensor_layer

N_SBC = 500  # test cases
param_names_list = list(PARAM_NAMES)
n_params = 8
ranks = np.zeros((N_SBC, n_params), dtype=int)

from cstr_sbi.luyben.priors import PRIOR_LOW_8D, PRIOR_HIGH_8D
rng = np.random.default_rng(SEED)

print(f'Running SBC on {N_SBC} test cases ...')
for i in range(N_SBC):
    # Draw true theta from prior
    theta_true = rng.uniform(PRIOR_LOW_8D, PRIOR_HIGH_8D).astype(np.float32)
    theta_jnp = jnp.array(theta_true)

    # Simulate one observation
    y0 = warm_start_ic(theta_jnp)
    proc_key, sens_key = jax.random.split(jax.random.PRNGKey(1000 + i))
    _, _, obs = simulate_em_window(theta_jnp, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0, key=proc_key)
    obs_noisy = apply_sensor_layer(obs, key=sens_key)
    t_out = jnp.arange(1, obs.shape[0] + 1) * 1.0
    s = np.asarray(compute_summary_statistics(obs_noisy, t_out))

    # Sample posterior
    samples = sample_posterior(posterior, s, n_samples=1000)

    # Compute rank of true theta in posterior samples
    for j in range(n_params):
        ranks[i, j] = int(np.sum(samples[:, j] < theta_true[j]))

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{N_SBC}')


In [ ]:
# KS test and rank histogram
fig, axes = plt.subplots(2, 4, figsize=(14, 6), constrained_layout=True)
for j, (ax, pname) in enumerate(zip(axes.ravel(), param_names_list)):
    r = ranks[:, j] / 1000.0
    ks_stat, ks_p = ks_1samp(r, uniform.cdf)
    ax.hist(r, bins=20, density=True, alpha=0.7)
    ax.axhline(1.0, ls='--', c='C1')
    ax.set_title(f'{pname}\nKS p={ks_p:.3f}')
    ax.set_xlabel('rank / N_samples')
print('SBC summary:')
for j, pname in enumerate(param_names_list):
    r = ranks[:, j] / 1000.0
    ks_stat, ks_p = ks_1samp(r, uniform.cdf)
    print(f'  {pname:10s}: KS p = {ks_p:.4f}')
fig.suptitle('SBC rank histograms -- Luyben 8 parameters')
plt.show()
